# Notebook 2: Build Open-World H5 Dataset

This notebook extracts both closed-world and open-world splits and builds `wfmeta_open_world_v1.h5` using the CPU executor (Accelerator = None).

In [ ]:
# Setup local working directories
!mkdir -p /kaggle/working/wf_split/closed
!mkdir -p /kaggle/working/wf_split/open
!mkdir -p /kaggle/working/h5_build

In [ ]:
# Clone/update Var-CNN repository from GitHub
import os
if not os.path.exists("/kaggle/working/Var-CNN"):
    !git clone https://github.com/ShortonKredit/Var-CNN.git /kaggle/working/Var-CNN
else:
    %cd /kaggle/working/Var-CNN
    !git pull origin main

In [ ]:
# Extract both Closed and Open splits to local Kaggle working SSD
print("Extracting closed_world_split.tar.gz...")
!tar -xzf /kaggle/input/wf-raw-splited-v1/closed_world_split.tar.gz -C /kaggle/working/wf_split/closed
print("Extracting open_world_split.tar.gz...")
!tar -xzf /kaggle/input/wf-raw-splited-v1/open_world_split.tar.gz -C /kaggle/working/wf_split/open
print("Extraction complete.")

In [ ]:
# Run H5 builder for Open-World scenario
!python /kaggle/working/Var-CNN/scripts/build_wfmeta_h5.py \
  --closed-dir /kaggle/working/wf_split/closed \
  --open-dir /kaggle/working/wf_split/open \
  --ranking-json /kaggle/working/Var-CNN/wfmeta/wfmeta_anova_ranked_features_v1.json \
  --output-dir /kaggle/working/h5_build \
  --seq-length 5000 \
  --build open_world

In [ ]:
# Inspect compiled H5 file
import h5py
h5_path = "/kaggle/working/h5_build/wfmeta_open_world_v1.h5"
if os.path.exists(h5_path):
    print("H5 File created successfully! Size:", os.path.getsize(h5_path)/(1024*1024), "MB")
    with h5py.File(h5_path, "r") as f:
        for split in f.keys():
            print(f"\nSplit: {split}")
            for ds in f[split].keys():
                print(f"  Dataset: {ds} | Shape: {f[split][ds].shape} | Dtype: {f[split][ds].dtype}")
else:
    print("Error: H5 file was not generated!")